In [1]:
import numpy as np
import pandas as pd

SEED = 42
SECONDS_PER_DAY = 24 * 60 * 60

def arrival_rate_per_second(hour, direction):
    morning = np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
    evening = np.exp(-0.5 * ((hour - 17) / 2) ** 2)
    if direction == "NS":
        per_5min = 8 + 35 * morning + 30 * evening
    else:
        per_5min = 6 + 25 * morning + 40 * evening
    return per_5min / 300   # vehicles per second

def generate_arrivals(seed, num_days=1):
    rng = np.random.default_rng(seed)
    t = np.arange(num_days * SECONDS_PER_DAY)
    hour = (t / 3600) % 24
    ns = rng.random(len(t)) < arrival_rate_per_second(hour, "NS")
    ew = rng.random(len(t)) < arrival_rate_per_second(hour, "EW")
    return ns.astype(int), ew.astype(int)

ns_arrivals, ew_arrivals = generate_arrivals(SEED)

print("Seconds simulated:", len(ns_arrivals))
print("Total NS vehicles:", ns_arrivals.sum())
print("Total EW vehicles:", ew_arrivals.sum())

# Sanity check: busy hour vs quiet hour
print("NS vehicles during 8 AM hour:", ns_arrivals[8*3600:9*3600].sum())
print("NS vehicles during 2 AM hour:", ns_arrivals[2*3600:3*3600].sum())

Seconds simulated: 86400
Total NS vehicles: 5636
Total EW vehicles: 5259
NS vehicles during 8 AM hour: 483
NS vehicles during 2 AM hour: 109


In [2]:
GREEN_FIXED = 30
YELLOW = 3
MIN_GREEN = 10
MAX_GREEN = 60
DISCHARGE_PER_SEC = 0.5   # one vehicle leaves every 2 seconds on green

def fixed_time_controller(state):
    return state["phase_timer"] >= GREEN_FIXED

def rule_based_controller(state):
    queue = state["ns_queue"] if state["phase"] == "NS_GREEN" else state["ew_queue"]
    t = state["phase_timer"]
    return t >= MAX_GREEN or (t >= MIN_GREEN and queue == 0)

def simulate(controller, ns_arrivals, ew_arrivals):
    n = len(ns_arrivals)
    ns_q, ew_q = 0, 0
    phase, phase_timer = "NS_GREEN", 0
    credit, served = 0.0, 0
    ns_log = np.zeros(n, dtype=int)
    ew_log = np.zeros(n, dtype=int)

    for t in range(n):
        # 1. arrivals
        ns_q += ns_arrivals[t]
        ew_q += ew_arrivals[t]

        # 2. discharge on green
        if phase in ("NS_GREEN", "EW_GREEN"):
            credit += DISCHARGE_PER_SEC
            if credit >= 1:
                credit -= 1
                if phase == "NS_GREEN" and ns_q > 0:
                    ns_q -= 1
                    served += 1
                elif phase == "EW_GREEN" and ew_q > 0:
                    ew_q -= 1
                    served += 1

        phase_timer += 1

        # 3. phase switching
        state = {"phase": phase, "phase_timer": phase_timer,
                 "ns_queue": ns_q, "ew_queue": ew_q, "t": t}
        if phase == "NS_GREEN" and controller(state):
            phase, phase_timer = "NS_YELLOW", 0
        elif phase == "NS_YELLOW" and phase_timer >= YELLOW:
            phase, phase_timer = "EW_GREEN", 0
        elif phase == "EW_GREEN" and controller(state):
            phase, phase_timer = "EW_YELLOW", 0
        elif phase == "EW_YELLOW" and phase_timer >= YELLOW:
            phase, phase_timer = "NS_GREEN", 0

        # 4. log
        ns_log[t], ew_log[t] = ns_q, ew_q

    # metrics
    hour = (np.arange(n) / 3600) % 24
    peak = (hour >= 8) & (hour < 9)
    total_arrivals = ns_arrivals.sum() + ew_arrivals.sum()
    total_delay = ns_log.sum() + ew_log.sum()   # vehicle-seconds spent waiting

    return {
        "avg_wait_sec_per_vehicle": round(total_delay / total_arrivals, 2),
        "avg_ns_queue_full_day": round(ns_log.mean(), 2),
        "avg_ew_queue_full_day": round(ew_log.mean(), 2),
        "avg_ns_queue_8am": round(ns_log[peak].mean(), 2),
        "avg_ew_queue_8am": round(ew_log[peak].mean(), 2),
        "max_ns_queue": int(ns_log.max()),
        "max_ew_queue": int(ew_log.max()),
        "vehicles_arrived": int(total_arrivals),
        "vehicles_served": int(served),
    }

print("Simulation engine ready.")

Simulation engine ready.


In [3]:
results = {
    "fixed_time": simulate(fixed_time_controller, ns_arrivals, ew_arrivals),
    "rule_based": simulate(rule_based_controller, ns_arrivals, ew_arrivals),
}

comparison = pd.DataFrame(results).T
comparison

,avg_wait_sec_per_vehicle,avg_ns_queue_full_day,avg_ew_queue_full_day,avg_ns_queue_8am,avg_ew_queue_8am,max_ns_queue,max_ew_queue,vehicles_arrived,vehicles_served
fixed_time,13.26,0.86,0.82,1.99,1.45,12.0,11.0,10895.0,10895.0
rule_based,7.50,0.49,0.45,1.18,0.92,7.0,7.0,10895.0,10895.0


In [5]:
import os
os.makedirs("../results", exist_ok=True)
comparison.to_csv("../results/controller_comparison_v2.csv")
print("Saved.")

Saved.


In [6]:
import joblib

ns_model = joblib.load("../models/ns_traffic_model.pkl")
ew_model = joblib.load("../models/ew_traffic_model.pkl")
feature_columns = joblib.load("../models/feature_columns.pkl")
ns_model.set_params(n_jobs=1)   # faster for single-row predictions
ew_model.set_params(n_jobs=1)

INTERVAL = 300        # 5 minutes in seconds
TOTAL_GREEN = 60      # green budget split between the two phases

class MLController:
    def __init__(self, ns_arr, ew_arr, day_of_week=2):
        self.ns_arr, self.ew_arr = ns_arr, ew_arr
        self.day_of_week = day_of_week       # 2 = a weekday
        self.last_k = 0
        self.ready = False
        self.target = {"NS_GREEN": GREEN_FIXED, "EW_GREEN": GREEN_FIXED}

    def _update(self, k):
        # k = number of completed 5-minute intervals (uses only past arrivals)
        if k < 13:                            # need 12 intervals of history
            return
        ns = self.ns_arr[:k * INTERVAL].reshape(k, INTERVAL).sum(axis=1)
        ew = self.ew_arr[:k * INTERVAL].reshape(k, INTERVAL).sum(axis=1)
        i = k - 1                             # the interval that just ended
        row = {
            "hour": ((i * INTERVAL) // 3600) % 24,
            "minute": (i * 5) % 60,
            "day_of_week": self.day_of_week,
            "is_weekend": int(self.day_of_week >= 5),
            "ns_vehicle_count": ns[i], "ew_vehicle_count": ew[i],
            "ns_rolling_mean_3": ns[i-3:i].mean(), "ew_rolling_mean_3": ew[i-3:i].mean(),
            "ns_rolling_mean_6": ns[i-6:i].mean(), "ew_rolling_mean_6": ew[i-6:i].mean(),
        }
        for lag in [1, 2, 3, 6, 12]:
            row[f"ns_lag_{lag}"] = ns[i - lag]
            row[f"ew_lag_{lag}"] = ew[i - lag]

        X = pd.DataFrame([row])[feature_columns]
        pred_ns = max(0, ns_model.predict(X)[0])
        pred_ew = max(0, ew_model.predict(X)[0])

        total = pred_ns + pred_ew
        share_ns = pred_ns / total if total > 0 else 0.5
        self.target["NS_GREEN"] = int(np.clip(round(TOTAL_GREEN * share_ns), MIN_GREEN, MAX_GREEN))
        self.target["EW_GREEN"] = int(np.clip(round(TOTAL_GREEN * (1 - share_ns)), MIN_GREEN, MAX_GREEN))
        self.ready = True

    def __call__(self, state):
        k = (state["t"] + 1) // INTERVAL
        if k > self.last_k:
            self.last_k = k
            self._update(k)

        if not self.ready:                    # first hour: behave like fixed-time
            return state["phase_timer"] >= GREEN_FIXED

        phase = state["phase"]
        queue = state["ns_queue"] if phase == "NS_GREEN" else state["ew_queue"]
        t = state["phase_timer"]
        return t >= self.target[phase] or (t >= MIN_GREEN and queue == 0)

ml_controller = MLController(ns_arrivals, ew_arrivals)
results["ml_based"] = simulate(ml_controller, ns_arrivals, ew_arrivals)
pd.DataFrame(results).T

,avg_wait_sec_per_vehicle,avg_ns_queue_full_day,avg_ew_queue_full_day,avg_ns_queue_8am,avg_ew_queue_8am,max_ns_queue,max_ew_queue,vehicles_arrived,vehicles_served
fixed_time,13.26,0.86,0.82,1.99,1.45,12.0,11.0,10895.0,10895.0
rule_based,7.50,0.49,0.45,1.18,0.92,7.0,7.0,10895.0,10895.0
ml_based,7.63,0.50,0.46,1.18,0.92,7.0,7.0,10895.0,10895.0


In [7]:
DISCHARGE_PER_SEC = 0.3   # lower capacity -> busier intersection

heavy = {
    "fixed_time": simulate(fixed_time_controller, ns_arrivals, ew_arrivals),
    "rule_based": simulate(rule_based_controller, ns_arrivals, ew_arrivals),
    "ml_based": simulate(MLController(ns_arrivals, ew_arrivals), ns_arrivals, ew_arrivals),
}
heavy_df = pd.DataFrame(heavy).T

DISCHARGE_PER_SEC = 0.5   # restore the normal setting
heavy_df

,avg_wait_sec_per_vehicle,avg_ns_queue_full_day,avg_ew_queue_full_day,avg_ns_queue_8am,avg_ew_queue_8am,max_ns_queue,max_ew_queue,vehicles_arrived,vehicles_served
fixed_time,73.53,1.97,7.30,14.90,2.54,28.0,88.0,10895.0,10895.0
rule_based,16.52,1.01,1.07,2.87,2.16,17.0,24.0,10895.0,10894.0
ml_based,25.59,1.50,1.73,2.86,3.11,26.0,33.0,10895.0,10894.0


In [8]:
DISCHARGE_PER_SEC = 0.3   # busy-intersection scenario

# 1) Tune TOTAL_GREEN on the seed-42 day only
tuning = {}
for tg in [60, 90, 120, 150]:
    TOTAL_GREEN = tg
    tuning[tg] = simulate(MLController(ns_arrivals, ew_arrivals),
                          ns_arrivals, ew_arrivals)["avg_wait_sec_per_vehicle"]
print("Avg wait by TOTAL_GREEN (seed 42):", tuning)
best_tg = min(tuning, key=tuning.get)
print("Best TOTAL_GREEN:", best_tg)

# 2) Test on days the tuning never saw
TOTAL_GREEN = best_tg
rows = []
for seed in [1, 2, 3]:
    ns_a, ew_a = generate_arrivals(seed)
    for name, ctrl in [("fixed_time", fixed_time_controller),
                       ("rule_based", rule_based_controller),
                       ("ml_based", MLController(ns_a, ew_a))]:
        rows.append({"seed": seed, "controller": name, **simulate(ctrl, ns_a, ew_a)})

test_df = pd.DataFrame(rows)
summary = test_df.groupby("controller")["avg_wait_sec_per_vehicle"].agg(["mean", "std"]).round(2)

DISCHARGE_PER_SEC = 0.5   # restore normal settings
TOTAL_GREEN = 60
summary

Avg wait by TOTAL_GREEN (seed 42): {60: np.float64(25.59), 90: np.float64(18.02), 120: np.float64(16.84), 150: np.float64(16.61)}
Best TOTAL_GREEN: 150


,mean,std
controller,,
fixed_time,83.78,9.28
ml_based,17.26,0.84
rule_based,17.15,0.83
